In [9]:
%load_ext autoreload
%autoreload 2

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from qiskit import QuantumCircuit, transpile
from qiskit_aer import StatevectorSimulator

import qlbm
from qlbm.physics import get_collision_diagonal, collision_nonuniform
from qlbm.circuits import encode, encode_links

sim = StatevectorSimulator()
cs  = 1 / np.sqrt(3)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [10]:
def uniform_vf(sites, u_mag=0.2):
    nd = len(sites)
    return np.stack([np.full(sites, u_mag / nd)] * nd)

def sinusoidal_vf(sites, u_mag=0.2):
    nd    = len(sites)
    grids = [np.linspace(0, 2 * np.pi, s) for s in sites]
    mesh  = np.meshgrid(*grids, indexing='ij')
    return np.stack([u_mag * np.sin(mesh[d % nd] + 2 * np.pi * d / nd) for d in range(nd)])

def alternating_vf(sites, u_mag=0.2):
    nd    = len(sites)
    grids = [np.arange(s) for s in sites]
    mesh  = np.meshgrid(*grids, indexing='ij')
    sign  = (-1) ** sum(mesh)
    return np.stack([u_mag * sign] * nd)

def run_collision(sites, links, weights, vf, optimization_level=2):
    """Encode -> encode_links -> collision.  Returns (max_error, collision_gate_count)."""
    num_links   = len(links)
    link_qubits = int(np.ceil(np.log2(max(num_links, 2))))
    max_links   = 2 ** link_qubits
    site_qubits = sum(int(np.ceil(np.log2(s))) for s in sites)
    num_sites   = 2 ** site_qubits
    num_qubits  = site_qubits + link_qubits + 1

    rng      = np.random.default_rng(42)
    rho0     = 0.3 + 0.4 * rng.random(sites)
    rho_flat = rho0.flatten(order='F')
    rho_norm = np.linalg.norm(rho_flat)

    diag      = get_collision_diagonal(link_qubits, links, weights, vf, cs)
    diag_norm = diag / np.max(np.abs(diag))

    coll_qc = QuantumCircuit(num_qubits)
    coll_qc.append(
        collision_nonuniform(site_qubits, link_qubits, diag).to_gate(),
        list(range(num_qubits)),
    )
    gate_count = sum(transpile(coll_qc, sim, optimization_level=optimization_level).count_ops().values())

    state = encode(rho0, link_qubits)
    qc    = QuantumCircuit(num_qubits)
    qc.initialize(state)
    qc.append(
        encode_links(link_qubits, num_links).to_gate(),
        list(range(site_qubits, num_qubits - 1)),
    )
    qc.append(
        collision_nonuniform(site_qubits, link_qubits, diag).to_gate(),
        list(range(num_qubits)),
    )
    sv = np.array(
        sim.run(transpile(qc, sim, optimization_level=optimization_level)).result().get_statevector()
    )

    sv_anc0      = np.real(sv[: num_sites * max_links])
    actual_sites = int(np.prod(sites))
    expected     = np.zeros(num_sites * max_links)
    for l in range(max_links):
        for s in range(actual_sites):
            expected[l * num_sites + s] = (
                diag_norm[l * actual_sites + s] * rho_flat[s] / (rho_norm * np.sqrt(max_links))
            )

    return np.abs(sv_anc0 - expected).max(), gate_count

In [11]:
# Experiment 1 – Does the velocity field structure affect collision precision? (D3Q27)
#
# D3Q27: all 27 velocity combinations in {-1,0,1}³.
# 27 links → 5 link qubits (padded to 32 slots).
# Grid sizes: 4³ and 8³.

def _d3q27():
    links, weights = [], []
    w = {0: 8/27, 1: 2/27, 2: 1/54, 3: 1/216}
    for vx in [-1, 0, 1]:
        for vy in [-1, 0, 1]:
            for vz in [-1, 0, 1]:
                links.append([vx, vy, vz])
                weights.append(w[abs(vx) + abs(vy) + abs(vz)])
    return dict(links=links, weights=weights)

D3Q27 = _d3q27()

results_3d = []

for n in [4, 8]:
    sites = (n, n, n)
    for vf_label, vf in [
        ('uniform',     uniform_vf(sites)),
        ('sinusoidal',  sinusoidal_vf(sites)),
        ('alternating', alternating_vf(sites)),
    ]:
        err, gc = run_collision(sites, D3Q27['links'], D3Q27['weights'], vf)
        results_3d.append(dict(n=n, vf=vf_label, error=err, gates=gc))
        print(f'D3Q27 {n}³  {vf_label:12s}  gates={gc:6d}  error={err:.2e}')

D3Q27 4³  uniform       gates=  4219  error=5.20e-17
D3Q27 4³  sinusoidal    gates=  5251  error=1.51e-06
D3Q27 4³  alternating   gates=  4339  error=7.98e-17
D3Q27 8³  uniform       gates= 32891  error=8.67e-18
D3Q27 8³  sinusoidal    gates= 35259  error=5.90e-07
D3Q27 8³  alternating   gates= 33011  error=3.12e-17


In [14]:
# Experiment 2 – Does the transpiler optimization level cause the precision loss? (D3Q27 4³)
#
# Three opt levels for the DiagonalGate path (sinusoidal), plus alternating and SVD at opt_level=2.

sites_wc  = (4, 4, 4)
vf_sin    = sinusoidal_vf(sites_wc, u_mag=0.2)
vf_alt    = alternating_vf(sites_wc, u_mag=0.2)

def run_collision_transpile_fn(sites, links, weights, vf, transpile_fn, use_svd=False):
    num_links   = len(links)
    link_qubits = int(np.ceil(np.log2(max(num_links, 2))))
    max_links   = 2 ** link_qubits
    site_qubits = sum(int(np.ceil(np.log2(s))) for s in sites)
    num_sites   = 2 ** site_qubits
    num_qubits  = site_qubits + link_qubits + 1

    rng      = np.random.default_rng(42)
    rho0     = 0.3 + 0.4 * rng.random(sites)
    rho_flat = rho0.flatten(order='F')
    rho_norm = np.linalg.norm(rho_flat)

    diag      = get_collision_diagonal(link_qubits, links, weights, vf, cs)
    diag_norm = diag / np.max(np.abs(diag))
    matrix    = np.diag(diag) if use_svd else diag

    state = encode(rho0, link_qubits)
    qc    = QuantumCircuit(num_qubits)
    qc.initialize(state)
    qc.append(
        encode_links(link_qubits, num_links).to_gate(),
        list(range(site_qubits, num_qubits - 1)),
    )
    qc.append(
        collision_nonuniform(site_qubits, link_qubits, matrix).to_gate(),
        list(range(num_qubits)),
    )
    qc_t = transpile_fn(qc)
    sv   = np.array(sim.run(qc_t).result().get_statevector())

    sv_anc0      = np.real(sv[: num_sites * max_links])
    actual_sites = int(np.prod(sites))
    expected     = np.zeros(num_sites * max_links)
    for l in range(max_links):
        for s in range(actual_sites):
            expected[l * num_sites + s] = (
                diag_norm[l * actual_sites + s] * rho_flat[s] / (rho_norm * np.sqrt(max_links))
            )
    return np.abs(sv_anc0 - expected).max(), sum(qc_t.count_ops().values())
opt0 = lambda qc: transpile(qc, sim, optimization_level=0)
opt1 = lambda qc: transpile(qc, sim, optimization_level=1)
opt2 = lambda qc: transpile(qc, sim, optimization_level=2)

opt_configs = [
    ('sinusoidal  opt_level=0',   opt0, vf_sin, False),
    ('sinusoidal  opt_level=1',   opt1, vf_sin, False),
    ('sinusoidal  opt_level=2',   opt2, vf_sin, False),
    ('alternating opt_level=2',   opt2, vf_alt, False),
    ('alternating opt_level=0',   opt0, vf_alt, False),
    ('alternating opt_level=1',   opt1, vf_alt, False),
    ('SVD         opt_level=2',   opt2,  vf_sin, True),
]

opt_results = []
print('D3Q27 4³  u_mag=0.2\n')
print(f'{"config":>30}  {"gates":>8}  {"error":>12}')
print('-' * 56)
for label, fn, vf, use_svd in opt_configs:
    err, gc = run_collision_transpile_fn(sites_wc, D3Q27['links'], D3Q27['weights'], vf, fn, use_svd)
    opt_results.append(dict(label=label, gates=gc, error=err))
    print(f'{label:>30}  {gc:>8d}  {err:>12.2e}')

D3Q27 4³  u_mag=0.2

                        config     gates         error
--------------------------------------------------------
       sinusoidal  opt_level=0     10084      1.13e-12
       sinusoidal  opt_level=1     10081      1.13e-12
       sinusoidal  opt_level=2      5257      1.51e-06
       alternating opt_level=2      4345      7.98e-17
       alternating opt_level=0      4348      1.04e-16
       alternating opt_level=1      4345      7.98e-17
Using SVD decomposition for non-diagonal collision matrix
Decomposition complete
Added manually-constructed controlled unitaries to circuit
       SVD         opt_level=2        10      6.94e-18
